# verify08d: ベクトル＋年齢・性別を「一緒に入力」→ トリアージ → F1評価

入力CSV（`評価用データ_label付け後.csv`）は **各ノードのcode（＝ベクトル）＋ 年齢 ＋ 性別** を含む。
このノートは **年齢・性別も入力としてそのまま一緒に読み**、`age` / `sex` としてトリアージへ渡す。

**verify08c との違い**
- 年齢を外部ファイル(`df_validation_input_renamed.csv`)から補完しない。**入力CSVの `年齢`/`性別` をそのまま使用**。
- `vector_to_triage(vec, age, sex)` に年齢・性別を一緒に渡す（性別ゲートの分岐も効く）。

**流れ**: 入力CSV → {ベクトル, 年齢, 性別} → `vector_to_triage` → main(VE/SE/LE) → F1(ja_dataset_v3.xlsx)
ベクトル入力装置(`vector_triage`)は無変更。年齢・性別は同じ入力機の中で一緒に読むだけ。

In [ ]:
# ===== セットアップ（Colab対応・モデル不要）=====
import os, sys, glob, json, csv, subprocess
IN_COLAB = 'google.colab' in sys.modules
REPO_URL, REPO_BRANCH = 'https://github.com/enenen13/Emergency_task', 'feature/headache-ablation-notebook'
DRIVE_ROOT_OVERRIDE = ''

DRIVE_ROOT = None
if IN_COLAB:
    from google.colab import drive
    try:
        drive.mount('/content/drive')
    except Exception as _e:
        print('mount retry:', _e); drive.mount('/content/drive', force_remount=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U',
                    'pyyaml', 'pandas', 'scikit-learn', 'openpyxl'], check=False)
    DRIVE_ROOT = DRIVE_ROOT_OVERRIDE or None
    if not os.path.isdir('Emergency_task'):
        subprocess.run(['git', 'clone', '--depth', '1', '-b', REPO_BRANCH, REPO_URL], check=False)
    else:
        subprocess.run(['git', '-C', 'Emergency_task', 'fetch', '--depth', '1', 'origin', REPO_BRANCH], check=False)
        subprocess.run(['git', '-C', 'Emergency_task', 'reset', '--hard', 'FETCH_HEAD'], check=False)
    REPO_DIR = os.path.abspath('Emergency_task')
else:
    REPO_DIR = os.path.abspath('.')
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# ★最新コードを確実に反映（ランタイム再起動 不要）:
#   git reset --hard で最新ファイルを取得済み。ただし既に import 済みのモジュールは
#   メモリに古いまま残るため、ここで捨てておく → 後続セルの import が最新を読む。
for _m in ('triage_pipeline', 'vector_triage', 'age_logic', 'feature_preprocess'):
    sys.modules.pop(_m, None)

MODE = 'vecinput_agesex'   # 出力ファイル名タグ

def _find_vec_csv():
    pats = []
    if DRIVE_ROOT:
        pats += [os.path.join(DRIVE_ROOT, '評価用データ_label付け後.csv')]
    pats += [os.path.join(REPO_DIR, '評価用データ_label付け後.csv'),
             '評価用データ_label付け後.csv',
             'dataset/評価用データ_label付け後.csv', 'train/評価用データ_label付け後.csv']
    for p in pats:
        h = sorted(glob.glob(p))
        if h:
            return h[0]
    return None
VEC_CSV = _find_vec_csv()
if VEC_CSV is None and IN_COLAB:
    from google.colab import files
    print('入力CSV（評価用データ_label付け後.csv）をアップロードしてください…')
    _up = files.upload()
    VEC_CSV = next((n for n in _up if n.lower().endswith('.csv')), None)
assert VEC_CSV, '入力CSV(評価用データ_label付け後.csv)が見つかりません'
print('REPO_DIR =', REPO_DIR, '| VEC_CSV =', VEC_CSV)

In [ ]:
# ===== 入力機: ベクトル ＋ 年齢 ＋ 性別 を「一緒に」読む =====
# ベクトル入力装置(vector_triage)は無変更。年齢/性別は同じCSVから一緒に読み、age/sexとして渡すだけ。
import pandas as pd
import vector_triage as vt, triage_pipeline as tp
NODE_MAP, NODE_ORDER = tp.NODE_MAP, vt.NODE_ORDER

df = pd.read_csv(VEC_CSV, dtype=str).fillna('')
assert 'id' in df.columns, 'id列が必要です'

node_cols = [c for c in df.columns if c in NODE_MAP]              # ベクトル列（NODE_MAP一致）
AGE_ALIASES = ('年齢', 'age', 'Age', 'AGE')
SEX_ALIASES = ('性別', 'sex', 'Sex', 'gender', 'Gender')
age_col = next((c for c in AGE_ALIASES if c in df.columns), None) # 年齢列
sex_col = next((c for c in SEX_ALIASES if c in df.columns), None) # 性別列
IGNORE = ('Conversation', 'conversation', 'Summary', 'summary')
skipped = [c for c in df.columns if c not in node_cols and c not in ('id',)
           and c not in (age_col, sex_col) and c not in IGNORE]
print(f'入力CSV {len(df)}行 / ベクトル列 {len(node_cols)} / 年齢列={age_col} / 性別列={sex_col} / 無視 {skipped[:6]}')

def _to_code(v):
    v = str(v).strip()
    if v == '':
        return None
    try:
        return int(float(v))
    except Exception:
        return None

def _meta(v):
    if v is None:
        return None
    s = str(v).strip()
    return None if (s == '' or s.lower() == 'nan') else s

recs = []
for _, r in df.iterrows():
    bp = {}
    for c in node_cols:
        cd = _to_code(r[c])
        if cd is None:
            continue
        bp[NODE_MAP[c]] = cd                                     # 列(yamlノード) → bertノード
    recs.append({'id': str(r['id']), 'bert_pred': bp,
                 'age': _meta(r[age_col]) if age_col else None,  # ★年齢も一緒に
                 'sex': _meta(r[sex_col]) if sex_col else None}) # ★性別も一緒に
r0 = recs[0]
print('例: id=%s age=%s sex=%s ノード数=%d' % (r0['id'], r0['age'], r0['sex'], len(r0['bert_pred'])))

In [ ]:
# ===== ベクトル(＋年齢・性別) → トリアージ ＋ サマリ =====
from collections import Counter
g = tp.load_graph()
results = []
for rec in recs:
    vec = vt.build_vector(rec['bert_pred'])                      # 入力装置はそのまま
    res = vt.vector_to_triage(vec, age=rec['age'], sex=rec['sex'], graph=g)  # 年齢・性別も一緒に渡す
    results.append({'id': rec['id'], 'age': rec['age'], 'sex': rec['sex'],
                    'main': res.main, 'sub1': res.sub1,
                    'common_completed': res.common_completed, 'common_stop': res.common_stop,
                    'completed': ','.join(res.completed_symptoms or []),
                    'furthest_broke': res.furthest_broke_symptom, 'report': res.report()})

for r in results[:3]:
    print('=' * 54); print('id=%s age=%s sex=%s' % (r['id'], r['age'], r['sex'])); print(r['report'])

print('\n==== サマリ ====')
print('main分布 :', dict(Counter(r['main'] for r in results)))
print('sub1分布 :', dict(Counter(r['sub1'] for r in results)))
cc = sum(1 for r in results if r['common_completed'])
print('共通フロー完了(症候別到達) : %d/%d' % (cc, len(results)))

## F1 / 混同行列（正解 = `ja_dataset_v3.xlsx` の Triage Label）

入力の `id` を `Case_id` と突き合わせ、`Triage Label`(Very/Semi/Low → VE/SE/LE) を正解に `main` を評価。
ローカルは `train/ja_dataset_v3.xlsx` 等を自動探索、Colabは無ければアップロード。

In [ ]:
# ===== F1 / 混同行列（正解=ja_dataset_v3.xlsx の Triage Label）=====
import os, glob, re
import pandas as pd

def _find_truth_xlsx():
    pats = []
    if DRIVE_ROOT:
        pats += [os.path.join(DRIVE_ROOT, 'ja_dataset_v3.xlsx'),
                 os.path.join(DRIVE_ROOT, 'dataset', 'ja_dataset_v3.xlsx')]
    pats += [os.path.join(REPO_DIR, 'train', 'ja_dataset_v3.xlsx'),
             os.path.join(REPO_DIR, 'dataset', 'ja_dataset_v3.xlsx'),
             'train/ja_dataset_v3.xlsx', 'dataset/ja_dataset_v3.xlsx', 'ja_dataset_v3.xlsx']
    for p in pats:
        h = sorted(glob.glob(p))
        if h:
            return h[0]
    return None

FORCE_UPLOAD_TRUTH = False    # ローカルはパス探索でOK。Colabで手動アップロードしたい時 True
TRUTH_XLSX = None if FORCE_UPLOAD_TRUTH else _find_truth_xlsx()
if TRUTH_XLSX is None:
    if IN_COLAB:
        from google.colab import files
        print('正解Excel（ja_dataset_v3.xlsx）をアップロードしてください…')
        _up = files.upload()
        TRUTH_XLSX = next((n for n in _up if n.lower().endswith(('.xlsx', '.xls'))), None)
    else:
        TRUTH_XLSX = _find_truth_xlsx()
assert TRUTH_XLSX, '正解Excel(ja_dataset_v3.xlsx)が見つかりません'
print('TRUTH_XLSX =', TRUTH_XLSX)

def _norm_label(s):
    t = re.sub(r'[\s_\-]+', '', str(s)).lower()
    if t.startswith('very'): return 'VE'
    if t.startswith('semi'): return 'SE'
    if t.startswith('low'):  return 'LE'
    return None

_xls = pd.ExcelFile(TRUTH_XLSX)
_sheet = 'ja' if 'ja' in _xls.sheet_names else _xls.sheet_names[0]
_df = _xls.parse(_sheet)
assert {'Case_id', 'Triage Label'} <= set(_df.columns), f'必要列なし: {list(_df.columns)}'
truth = {}
for _, r in _df.iterrows():
    lab = _norm_label(r['Triage Label'])
    if lab:
        truth[str(r['Case_id'])] = lab
print('正解 %d 件 分布: %s' % (len(truth), {v: sum(x == v for x in truth.values()) for v in ['VE', 'SE', 'LE']}))

pairs = [(str(r['id']), r['main']) for r in results if str(r['id']) in truth]
y_true = [truth[i] for i, _ in pairs]
y_pred = [p for _, p in pairs]
print('突き合わせ %d/%d 件' % (len(pairs), len(results)))
_other = sum(1 for p in y_pred if p not in ('VE', 'SE', 'LE'))
if _other:
    print('  ※ main が VE/SE/LE 以外(複数完了 等): %d件 → 誤り計上' % _other)

labels = ['VE', 'SE', 'LE']
from sklearn.metrics import classification_report, confusion_matrix, f1_score
print('\n===== F1 =====')
for avg in ('macro', 'micro', 'weighted'):
    print('%8s-F1 : %.4f' % (avg, f1_score(y_true, y_pred, labels=labels, average=avg, zero_division=0)))
print('\n', classification_report(y_true, y_pred, labels=labels, zero_division=0))
cm = confusion_matrix(y_true, y_pred, labels=labels)
print('混同行列 (行=正解, 列=予測)  順:', labels)
print('        ' + '  '.join('%4s' % l for l in labels))
for lab, row in zip(labels, cm):
    print('  %4s | ' % lab + '  '.join('%4d' % v for v in row.tolist()))

In [ ]:
# ===== 混同行列を「正解◯件中◯件」で表示（複数完了も隠さず1列に）=====
CLASSES = ['VE', 'SE', 'LE']
def _bucket(p):
    return p if p in CLASSES else '他(複数完了等)'
pred_cols = CLASSES + ['他(複数完了等)']

tbl = {t: {c: 0 for c in pred_cols} for t in CLASSES}
for i, p in pairs:                       # pairs=[(id, pred_main)]（F1セルで作成済み）
    tbl[truth[i]][_bucket(p)] += 1

print('=== 行=正解 / 各セル: 件数(その正解の中での割合) ===')
print('正解＼予測  ' + '  '.join(f'{c:>13}' for c in pred_cols) + '      計')
for t in CLASSES:
    total = sum(tbl[t].values())
    cells = []
    for c in pred_cols:
        n = tbl[t][c]; pct = (n / total * 100) if total else 0
        cells.append(f'{n:>3}件({pct:4.1f}%)')
    print(f'  {t:<6} ' + '  '.join(f'{x:>13}' for x in cells) + f'   {total:>3}件')

print('\n=== recall: 正解◯件中◯件を正しく当てた ===')
for t in CLASSES:
    total = sum(tbl[t].values()); hit = tbl[t][t]
    print(f'  {t}: {total}件中 {hit}件を{t}と予測 → {hit / total * 100:.1f}%' if total else f'  {t}: 0件')

print('\n=== precision: ◯と予測した中で本当に◯だった ===')
for c in CLASSES:
    predn = sum(tbl[t][c] for t in CLASSES); hit = tbl[c][c]
    print(f'  {c}と予測 {predn}件中 {hit}件が本当に{c} → {hit / predn * 100:.1f}%' if predn else f'  {c}と予測 0件')

## 遷移トレース ＆ 到達度の可視化

`recs`（年齢・性別込み）と同じ入力で計算するので、上のF1結果と整合します。

- **追加①** 各通報が共通フローのどのノードを通り、どこで止まった/症候別へ抜けたか。
- **追加②** 「共通 完了/途切れ × 症状 完了/途切れ」の4象限で、どのトリアージになるかサンプル確認。
- **追加③** 症状ごとの到達度（遷移数）と、共通完了時の症状 完了/途切れ 本数・割合。

In [ ]:
# ===== 追加①: 共通フローの遷移トレース =====
import pandas as pd
from collections import Counter

def _answers(rec):
    ans = dict(vt.DEFAULT_BASE_ANSWERS)
    ans.update(tp.build_answers_from_bert(rec['bert_pred'], g, age=rec.get('age'), sex=rec.get('sex')))
    if rec.get('age') is not None and tp.age_logic is not None:
        try:
            ans = tp.age_logic.augment_answers_with_age(ans, rec['age'], g.index)
        except Exception:
            pass
    return ans

def common_trace(rec):
    ans = _answers(rec)
    cw = tp.traverse_common(g, ans)
    completed = cw.stop.startswith('reached:') or cw.stop.startswith('route_to')
    return {'id': rec['id'], 'common_path': ' → '.join(cw.path),
            'last_node': cw.path[-1] if cw.path else '', 'stop': cw.stop,
            'route': cw.route or '', 'common_completed': completed}

ctrace = [common_trace(r) for r in recs]
ctdf = pd.DataFrame(ctrace)
print('=== 共通フローが「どこで止まった/抜けたか」分布（last_node / stop）===')
for (ln, st), n in ctdf.groupby(['last_node', 'stop']).size().sort_values(ascending=False).items():
    print(f'  {ln:<24} {st:<26} {n:>4} 件 ({n / len(ctdf) * 100:4.1f}%)')
print(f'\n共通完了(症候別到達): {int(ctdf.common_completed.sum())}/{len(ctdf)} '
      f'({ctdf.common_completed.mean() * 100:.1f}%)')
print('\n=== サンプル: 先頭8件が共通のどこを通ったか ===')
for row in ctrace[:8]:
    mark = '完了' if row['common_completed'] else '途切れ'
    tail = f'  → [route_to:{row["route"]}]' if row['route'] else ''
    print(f'id={row["id"]:>4} [{mark}] stop={row["stop"]}')
    print(f'   {row["common_path"]}{tail}')
ctdf.head(10)

In [ ]:
# ===== 追加②: 4象限（共通 完了/途切れ × 症状 完了/途切れ）→ トリアージ サンプル =====
from collections import defaultdict

def full_detail(rec):
    ans = _answers(rec)
    cw = tp.traverse_common(g, ans)
    common_completed = cw.stop.startswith('reached:') or cw.stop.startswith('route_to')
    comp, broke = [], []
    for pid in g.protocol_ids:
        w = tp.traverse_symptom(g, pid, ans)
        if not any(n in ans for n in w.path):
            continue
        if w.triage and not w.broke_off:
            comp.append({'symptom': pid, 'triage': w.triage, 'transitions': w.transitions})
        else:
            broke.append({'symptom': pid, 'stop': w.stop, 'transitions': w.transitions})
    res = vt.vector_to_triage(vt.build_vector(rec['bert_pred']), age=rec.get('age'),
                              sex=rec.get('sex'), graph=g)
    return {'id': rec['id'], 'common_completed': common_completed, 'sym_completed': len(comp) > 0,
            'completed': comp, 'broke': broke, 'main': res.main, 'sub1': res.sub1, 'reason': res.reason}

details = [full_detail(r) for r in recs]
QNAME = {(True, True):  '① 共通完了 × 症状完了  → 症状のトリアージを採用(最優先)',
         (True, False): '② 共通完了 × 症状途切れ → 途切れVE(安全側)',
         (False, True): '③ 共通途切れ × 症状完了 → 途切れVE(安全側)',
         (False, False):'④ 共通途切れ × 症状途切れ → 途切れVE(安全側)'}
quad = defaultdict(list)
for d in details:
    quad[(d['common_completed'], d['sym_completed'])].append(d)
print('=== 4象限の件数・割合 ===')
for k in [(True, True), (True, False), (False, True), (False, False)]:
    n = len(quad[k]); print(f'  {QNAME[k]:<44} : {n:>4} 件 ({n / len(details) * 100:4.1f}%)')
print('\n=== 各象限のサンプル ===')
for k in [(True, True), (True, False), (False, True), (False, False)]:
    lst = quad[k]
    print(f'\n[{QNAME[k]}]  ({len(lst)}件)')
    if not lst:
        print('   該当なし'); continue
    d = lst[0]
    comp = ', '.join(f"{c['symptom']}:{c['triage']}({c['transitions']}歩)" for c in d['completed']) or 'なし'
    brk = ', '.join(f"{b['symptom']}({b['transitions']}歩)" for b in d['broke'][:5]) or 'なし'
    print(f"   例 id={d['id']}  →  main={d['main']} / sub1={d['sub1']}")
    print(f"      完了症状 : {comp}")
    print(f"      途切れ症状: {brk}")
    print(f"      根拠     : {d['reason']}")

In [ ]:
# ===== 追加③: 症状の到達度 ＋ 共通完了時の 進む/途切れ 本数・割合 =====
import pandas as pd
rows = []
for d in details:
    for c in d['completed']:
        rows.append({'id': d['id'], 'symptom': c['symptom'], 'status': '完了',
                     'transitions': c['transitions'], 'common_completed': d['common_completed']})
    for b in d['broke']:
        rows.append({'id': d['id'], 'symptom': b['symptom'], 'status': '途切れ',
                     'transitions': b['transitions'], 'common_completed': d['common_completed']})
sdf = pd.DataFrame(rows)
n_all = len(sdf)
print(f'=== 関与した症状レコード（回答が乗った症状 × 通報）: {n_all} 本 ===')
for s in ['完了', '途切れ']:
    n = int((sdf.status == s).sum()); print(f'  {s}: {n} 本 ({n / n_all * 100:.1f}%)')
print('\n=== 「共通をたどれた(共通完了)」通報に限定：症状はどこまで進むか ===')
cc = sdf[sdf.common_completed]
if len(cc):
    for s in ['完了', '途切れ']:
        n = int((cc.status == s).sum()); print(f'  {s}: {n} 本 / {len(cc)} 本中 ({n / len(cc) * 100:.1f}%)')
    print('\n  到達度(遷移数=どれだけ深く辿れたか) の統計:')
    print(cc.groupby('status')['transitions'].agg(['count', 'mean', 'median', 'max']).round(2).to_string())
print('\n=== 症状別: 完了/途切れ 本数と完了率（本数降順）===')
piv = sdf.pivot_table(index='symptom', columns='status', values='id', aggfunc='count', fill_value=0)
for c in ['完了', '途切れ']:
    if c not in piv.columns:
        piv[c] = 0
piv['計'] = piv['完了'] + piv['途切れ']
piv['完了率%'] = (piv['完了'] / piv['計'] * 100).round(1)
print(piv.sort_values('計', ascending=False)[['完了', '途切れ', '計', '完了率%']].to_string())

In [ ]:
# ===== 出力: 1行1通報のトリアージ詳細CSV（年齢・性別込み）=====
import csv, os
os.makedirs('output', exist_ok=True)
out_detail = f'output/vecinput_triage_{MODE}.csv'
with open(out_detail, 'w', encoding='utf-8-sig', newline='') as f:
    w = csv.writer(f)
    w.writerow(['id', 'age', 'sex', 'main', 'sub1', 'common_completed', 'common_stop', 'completed', 'furthest_broke'])
    for r in results:
        w.writerow([r['id'], r['age'], r['sex'], r['main'], r['sub1'],
                    r['common_completed'], r['common_stop'], r['completed'], r['furthest_broke']])
print('詳細CSV →', out_detail, f'({len(results)}行)')